# BME688-Experimente -- Analyse & Plots

Dieses Notebook enthält **keine ML-Logik mehr**. Es ruft nur noch
`ml.pipeline.run_experiment()` auf und kümmert sich um Visualisierung und
Modellvergleich. Die eigentliche Pipeline (Daten laden, splitten,
skalieren, trainieren, auswerten, tracken) lebt in `ml/` und ist über
`pytest` unabhängig vom Notebook testbar.

Architektur und Erweiterungshinweise: siehe `ARCHITEKTUR.md`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from ml.config import ExperimentConfig
from ml.pipeline import run_experiment
from ml.tracking import load_results
from ml.models.registry import available_models

print("Verfügbare Modelle:", available_models())

## Experiment konfigurieren & ausführen

Jede Zelle unten ist ein Experiment: Config bauen, `run_experiment()`
aufrufen, Ergebnis-Objekt bekommen. Für ein neues Modell reicht ein
neuer `model_name` (siehe `available_models()` oben) -- kein Code in
`ml/` muss dafür angefasst werden.

In [ ]:
config_rf = ExperimentConfig(
    data_dir="/home/tun/Projects/tuc/ML/Forschungspraktikum/CSV/transformiert/Badset/level2_per_profile",   # Pfad zu den core.py-Feature-CSVs anpassen
    feature_level="Level 2",
    feature_set="resistance_gassensor",
    imputation="Median",
    scaling=True,
    model_name="random_forest",
    model_params={"n_estimators": 300},
    notes="Baseline",
)

result_rf = run_experiment(config_rf)
print(f"Experiment-ID: {result_rf.experiment_id}")
print(result_rf.evaluation.classification_report_text)

## Plots für ein Experiment

Confusion Matrix und (falls vom Modell unterstützt) Feature Importance
werden hier -- und nur hier -- gezeichnet, nicht in `ml/`. Die PNGs
landen direkt in `result.exp_dir`, also im selben Ordner wie
`config.json` und `report.txt` aus dem Tracking-Modul.

In [ ]:
def plot_confusion_matrix(result):
    ev = result.evaluation
    fig, ax = plt.subplots(figsize=(6, 6))
    sns.heatmap(ev.confusion_matrix, annot=True, fmt="d", cmap="Blues",
                xticklabels=ev.labels, yticklabels=ev.labels, ax=ax)
    ax.set_xlabel("Vorhergesagt")
    ax.set_ylabel("Tatsächlich")
    ax.set_title(f"Confusion Matrix -- {result.config.model_name}")
    plt.tight_layout()
    fig.savefig(result.exp_dir / "confusion_matrix.png", dpi=150)
    plt.show()


def plot_feature_importance(result, top_n=20):
    importances = result.evaluation.feature_importances
    if importances is None:
        print(f"{result.config.model_name} unterstützt keine feature_importances_.")
        return
    fig, ax = plt.subplots(figsize=(8, 6))
    importances.head(top_n).iloc[::-1].plot.barh(ax=ax)
    ax.set_xlabel("Feature Importance")
    ax.set_title(f"Top Features -- {result.config.model_name}")
    plt.tight_layout()
    fig.savefig(result.exp_dir / "feature_importance.png", dpi=150)
    plt.show()


plot_confusion_matrix(result_rf)
plot_feature_importance(result_rf)

## Weitere Modelle

In [ ]:
config_svm = ExperimentConfig(
    data_dir="data/level2_per_profile",
    feature_level="Level 2",
    feature_set="all_features",
    imputation="Median",
    scaling=True,   # fuer SVM wichtig
    model_name="svm",
    model_params={"C": 1, "kernel": "rbf"},
    notes="",
)

result_svm = run_experiment(config_svm)
plot_confusion_matrix(result_svm)

In [ ]:
config_knn = ExperimentConfig(
    data_dir="data/level2_per_profile",
    feature_level="Level 2",
    feature_set="all_features",
    imputation="Median",
    scaling=True,
    model_name="knn",
    model_params={"n_neighbors": 5},
    notes="",
)

result_knn = run_experiment(config_knn)
plot_confusion_matrix(result_knn)

## Modellvergleich

Alle bisher gelaufenen Experimente -- über beliebig viele
Notebook-Sitzungen hinweg -- stehen in `experiment_results.csv`.

In [ ]:
history = load_results(config_rf.result_file)
history.sort_values("F1", ascending=False)[
    ["Experiment ID", "Model", "Feature Level", "Feature Set", "Scaling",
     "Accuracy", "Precision", "Recall", "F1", "False Positive Rate", "Notes"]
]